In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'

donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
leiden_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(leiden_path, sep='\t', index_col=0)
leiden.shape

In [ ]:
leiden.head()

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', index_col=0)
ips_clusters.shape

In [ ]:
ips_clusters.head()

In [ ]:
adatas = []
for d in donors:
    donor_adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/{d}.h5ad')
    donor_adata.obs['cluster'] = clusters.reindex(donor_adata.obs.index)['pred_coarse']
    donor_adata.obs['ips_clusters'] = ips_clusters.reindex(donor_adata.obs.index)['ips_subtype']
    adatas.append(donor_adata)

In [ ]:
for donor_adata in adatas:
    sc.pl.umap(donor_adata, color=['ips_clusters', 'leiden', 'cluster'], ncols=2)

In [ ]:
for donor_adata in adatas:
    sc.pl.umap(donor_adata, color=['ips_clusters', 'leiden'], ncols=2)

In [ ]:
tmp_umaps = []
for i,donor_adata in enumerate(adatas):
    d = donors[i]
    umap_df = pd.DataFrame(index=donor_adata.obs.index)
    umap_df['umap0'] = donor_adata.obsm['X_umap'][:,0]
    umap_df['umap1'] = donor_adata.obsm['X_umap'][:,1]
    umap_df['cluster'] = clusters.reindex(umap_df.index)['pred_coarse']
    umap_df['ips_subtype'] = ips_clusters.reindex(umap_df.index)['ips_subtype']
    umap_df['donor'] = d
    tmp_umaps.append(umap_df)

In [ ]:
merged = pd.concat(tmp_umaps)
merged.shape

In [ ]:
merged.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
merged.to_csv(f'{indir}/snmct_snm3c_mcg_ips_subtype_umap_per_donor.csv', sep='\t')

### merged UMAP

In [ ]:
# small_adata.write_h5ad(f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.small.h5ad')

In [ ]:
adata_path = f'{projdir}/h5ad/integrate_m3c/merged.processed.harmony.small.h5ad'
adata = sc.read_h5ad(adata_path)

In [ ]:
sc.pl.umap(adata)

In [ ]:
tmp_index = [x.replace('-0', '').replace('-1', '') for x in adata.obs.index]
umap_df = pd.DataFrame(index=tmp_index)
umap_df['umap0'] = adata.obsm['X_umap'][:,0]
umap_df['umap1'] = adata.obsm['X_umap'][:,1]

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/xgboost_time_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', index_col=0)
ips_clusters.shape

In [ ]:
umap_df['cluster'] = clusters.reindex(umap_df.index)['cluster']
umap_df['donor'] = clusters.reindex(umap_df.index)['line']
umap_df['time'] = clusters.reindex(umap_df.index)['time']
umap_df['ips_subtype'] = ips_clusters.reindex(umap_df.index)['ips_subtype']

In [ ]:
umap_df.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
umap_df.to_csv(f'{indir}/snmct_snm3c_mcg_ips_subtype_umap.csv', sep='\t')

In [ ]:
!date